## ***1. Установка библиотеки ***

In [ ]:
%pip install earthengine-api certifi

Note: you may need to restart the kernel to use updated packages.


## ***2. Аутентификация через Python-код 🔑***

In [4]:
import ee
import ssl
import certifi
import os
import urllib.request

# Настройка SSL-сертификатов для macOS
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# Создаем SSL контекст с правильными сертификатами
ssl_context = ssl.create_default_context(cafile=certifi.where())

# Патчим urllib.request для использования правильных сертификатов
original_urlopen = urllib.request.urlopen
def patched_urlopen(*args, **kwargs):
    if 'context' not in kwargs:
        kwargs['context'] = ssl_context
    return original_urlopen(*args, **kwargs)
urllib.request.urlopen = patched_urlopen

# Запуск процесса авторизации
ee.Authenticate()


Successfully saved authorization token.


## ***3. Авторизация и инициализация***

In [5]:
ee.Initialize(project='my-project-mnad-gis')

## ***4.Создание функции для расчета NDVI***

In [6]:
def calculate_ndvi(image):
    # Используем встроенный метод для формулы (B8-B4)/(B8+B4)
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    # Возвращаем исходный снимок, добавив к нему новый слой NDVI
    return image.addBands(ndvi)

## ***5. Запрос данных для конкретной локации***

In [7]:
def get_location_score(lat, lon):
    # 1. Создаем точку
    point = ee.Geometry.Point([lon, lat])
    
    # 2. Запрашиваем коллекцию Sentinel-2
    data = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(point)
            .filterDate('2023-06-01', '2023-06-30')
            # Сортируем по облачности, чтобы взять самый чистый снимок
            .sort('CLOUDY_PIXEL_PERCENTAGE')
            .first()) # Берем первый (самый лучший) снимок
    
    # 3. Считаем NDVI
    image_with_ndvi = calculate_ndvi(data)
    
    # 4. Вычисляем среднее значение в радиусе 500 метров от точки
    stats = image_with_ndvi.select('NDVI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point.buffer(500),
        scale=10 # Разрешение спутника Sentinel-2 (10 метров на пиксель)
    ).getInfo()
    
    return stats['NDVI']

## ***Шаг 4: Как это использовать в Streamlit***

In [8]:
# Пример вызова для Москвы (район Хамовники)
# lat, lon = 55.72, 37.57
lat, lon = 58.693560, 56.064610
score = get_location_score(lat, lon)

print(f"Индекс озеленения района: {score:.2f}")
# Если score > 0.5 — это "зеленый рай"
# Если score < 0.2 — "каменные джунгли"


Индекс озеленения района: -0.08
